In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
df = spark.read.table("yipidata.bronze.tech_news")

In [0]:
df.display()

In [0]:
df.select('category').distinct().display()

In [0]:
df = df.withColumn('category_new', when((col('category') ==  'Financial Technology') | (col('category') == 'FinTech'), 'Financial Technology')\
                              .when((col('category') == 'Software') | (col('category') == 'Enterprise Software'), 'Software')\
                              .when(col('category') == 'SaaS', 'SaaS')\
                              .when((col('category') == 'Analytics') | (col('category') == 'Data Analytics'), 'Analytics')\
                              .when((col('category') == 'AI/ML') | (col('category') == 'Artificial Intelligence') |(col('category') == 'AI & ML') | (col('category') == 'Machine Learning'), 'Artificial Intelligence')\
                              .when((col('category') == 'Security') | (col('category') == 'Cybersecurity') | (col('category') == 'InfoSec'), 'Cybersecurity')\
                              .when(col('category') == 'Finance', 'Finance')\
                              .when((col('category') == 'Cloud Computing') | (col('category') == 'Cloud') | (col('category') == 'Big Data') | (col('category') == 'Cloud Services'), 'Cloud Computing')\
                              .otherwise('')
              )

In [0]:
df.select('author').fillna('N/A').display()

In [0]:
df = df.withColumnRenamed('revenue', 'revenue_raw')

In [0]:
df = df.withColumn('multiplier',  when(lower(col('revenue_raw')).rlike('m|million'), 1000000)\
                             .when(lower(col('revenue_raw')).rlike('b|billion'), 1000000000)\
                             .otherwise(1)  
              )

In [0]:
df.display()

In [0]:
df = df.withColumn('currency', when(col('revenue_raw').rlike(r'€|EUR'), '€')\
                          .when(col('revenue_raw').rlike(r'£|GBP'), '£')\
                          .when(col('revenue_raw').rlike(r'¥|JPY'), '¥')\
                          .when(col('revenue_raw').rlike(r'\$|USD'), '$')\
                          .otherwise('')         
              )

In [0]:
df.select('currency', "revenue_raw").where(col('currency') != '')\
                                 .display()


In [0]:
df.display()

In [0]:
df = df.withColumn(
    "revenue_extr_1",
    regexp_replace(
        regexp_extract(
            col("revenue_raw"),
            r"(\d+(?:,\d{3})*(?:\.\d+)?)",
            1
        ),
        ",",
        ""
    )
)

In [0]:
df.display()

In [0]:
df = df.withColumn('revenue_extr_1', when(col("revenue_extr_1") != '',
                                    col("revenue_extr_1").cast('double')))


In [0]:
df = df.withColumn(
    "revenue_extr_2",
    when(
        col("revenue_raw").rlike(r"\s*[-–]\s*"),
        regexp_replace(
            regexp_extract(
                col("revenue_raw"),
                r"[-–][^\d]*(\d+(?:,\d{3})*(?:\.\d+)?)",
                1
            ),
            ",",
            ""
        ).cast("double")
    )
)


In [0]:
df = df.withColumn('revenue_num', when(col('revenue_extr_2').isNotNull(), (col("revenue_extr_1") + col("revenue_extr_2"))/2)\
                           .otherwise(col("revenue_extr_1")))

In [0]:
df.select('multiplier', "revenue_extr_1", "revenue_extr_2", "revenue_num", (col('revenue_num') * col('multiplier')).alias('revenue')).display()


In [0]:
df = df.withColumn('exc_rate_mul', when(col('currency') == '$', lit('1'))\
                                   .when(col('currency') == '€', lit('1.1'))\
                                   .when(col("currency") == '£', lit('1.27'))\
                                   .when(col("currency") == '¥', lit('0.00667'))\
              )

In [0]:
df = df.withColumn('revenue', col('revenue_num') * col('multiplier') * col('exc_rate_mul'))

In [0]:
df.select("multiplier", 'currency', 'revenue_num', "exc_rate_mul", 'revenue').display()

In [0]:
#cast revenue column to integer

In [0]:
df = df.withColumn('revenue', round(col('revenue')))

In [0]:
df = df.withColumn('date_cleaned', coalesce(
    try_to_date('published_date', 'dd MMM yyyy'),
    try_to_date('published_date', 'MM/dd/yyyy'),
    try_to_date('published_date', 'yyyy-MM-dd'),
    try_to_date('published_date', 'MMMM dd, yyyy'),
    try_to_date('published_date', "yyyy-MM-dd'T'HH:mm:ss'Z'"),
    try_to_date('published_date', 'dd-MM-yyyy'),
    try_to_date('published_date', 'MM-dd-yyyy')
))

In [0]:
df = df.withColumn('year', year('date_cleaned'))\
    .withColumn('month', monthname('date_cleaned'))\
    .withColumn('quarter', concat(lit('Q'), quarter('date_cleaned')))

In [0]:
df.filter(col('year') == 2020).display()

In [0]:
df.select('company_name').distinct().display()

In [0]:
df = df.withColumn('company_name_new', when(lower(col('company_name')).rlike('aws|amazon web services (aws)'), 'Amazon Web Services')\
                                .when(col('company_name') == 'Databricks Inc.', 'Databricks')\
                                .when(col('company_name') == 'Palantir Technologies', 'Palantir')\
                                .when(col('company_name') == 'Snowflake Inc.', 'Snowflake')\
                                .when(col('company_name') == 'Stripe Inc.', 'Stripe')\
                                .when(col('company_name').contains('SpaceX'), 'SpaceX')\
                                .when(col('company_name').rlike('Microsoft Azure|Azure'), 'Microsoft')\
                                .when(lower(col('company_name')).contains('deepmind'), 'Google DeepMind')\
                                .when(lower(col('company_name')).rlike('nvidia|nvidia corporation'), 'NVIDIA')\
                                .when(lower(col('company_name')).rlike('open ai|openai inc.'), 'OpenAI')\
                                .when(lower(col('company_name')).rlike('cloudflare'), 'Cloudflare')\
                                .when(lower(col('company_name')).rlike('mongo db|mongodb'), 'MongoDB')
                                .when(lower(col('company_name')).rlike('facebook ai research|meta ai research'), 'Meta AI')\
                                .when(lower(col('company_name')).rlike('data robot|datarobot'), 'DataRobot')\
                                .otherwise(col('company_name'))
    )

In [0]:
df.select('company_name_new').distinct().display()

In [0]:
df.display()

In [0]:
df = df.drop("_rescued_data", "multiplier", "category", "company_name","exc_rate_mul", "revenue_extr_1", "revenue_extr_2", "revenue_num", "published_date", "revenue_raw", "currency")

In [0]:
df.display()

In [0]:
df = df.withColumn('word_count', col('word_count').cast('double'))

In [0]:
df = df.withColumn('word_count', col('word_count').cast('integer'))

In [0]:
df.display()

In [0]:
#use a merge here
#merge on article_id

In [0]:
#join both tables on company, still at the silver level.

In [0]:
df.createOrReplaceTempView("source_view")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS yipidata.silver.company_articles (
    article_id STRING,
    title STRING,
    summary STRING,
    `url` STRING,
    author STRING,
    word_count INT,
    category_new STRING,
    revenue DOUBLE,
    date_cleaned DATE,
    `year` INT,
    `month` STRING,
    `quarter` STRING,
    company_name_new STRING
)
USING DELTA;

In [0]:
%sql
MERGE INTO yipidata.silver.company_articles AS trg
USING source_view AS src
ON trg.article_id = src.article_id
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *

In [0]:
%sql
SELECT * FROM yipidata.silver.company_articles